# 🏡 Aetheris ArchViz AI Studio — Google Colab GPU Server (T4 15GB Miễn Phí)
### ⚡ Hướng dẫn 1-Click:
1. Chọn menu **Thời gian chạy (Runtime)** > **Thay đổi loại thời gian chạy (Change runtime type)** > Chọn **T4 GPU** > Bấm **Lưu**.
2. Bấm nút **Play ▶ (Chạy)** ở ô bên dưới.
3. Copy link `https://xxxx.trycloudflare.com` xuất hiện ở cuối và dán vào Web App!

In [ ]:
#@title 🚀 1-CLICK RUN: Khởi Động ComfyUI GPU Cloud Server
import os, subprocess, time, re, urllib.request
from IPython.display import display, HTML, clear_output

print("⏳ [1/4] Đang cài đặt môi trường GPU & Aria2 siêu tốc...")
%cd /content
!apt-get update -qq && apt-get install -y -qq aria2 > /dev/null 2>&1

# 1. Clone ComfyUI Core nếu chưa có
if not os.path.exists("/content/ComfyUI"):
    print("⚡ Đang tải ComfyUI Engine...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI > /dev/null 2>&1

# 2. Clone Custom Nodes thiết yếu
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-Manager"):
    !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git > /dev/null 2>&1
if not os.path.exists("comfyui_controlnet_aux"):
    !git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux.git > /dev/null 2>&1

# 3. Cài đặt Python Dependencies
print("📦 [2/4] Đang cài đặt PyTorch CUDA...")
%cd /content/ComfyUI
!pip install -q -r requirements.txt > /dev/null 2>&1
!pip install -q requests pillow > /dev/null 2>&1

# 4. Cài đặt Cloudflare Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 5. Tải Models Realistic Vision V5.1 & ControlNet Depth
print("📥 [3/4] Đang tải AI Models kiến trúc đa luồng (Realistic Vision & ControlNet Depth)...")
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/controlnet", exist_ok=True)

rv_path = "/content/ComfyUI/models/checkpoints/Realistic_Vision_V5.1.safetensors"
if not os.path.exists(rv_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/checkpoints -o Realistic_Vision_V5.1.safetensors https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors > /dev/null 2>&1

cn_path = "/content/ComfyUI/models/controlnet/control_v11f1p_sd15_depth.pth"
if not os.path.exists(cn_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/controlnet -o control_v11f1p_sd15_depth.pth https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth > /dev/null 2>&1

# 6. Khởi chạy ComfyUI nền & Đảm bảo Server sẵn sàng trước khi mở Tunnel
print("⚡ [4/4] Đang khởi động ComfyUI GPU Server...")
!fuser -k 8188/tcp > /dev/null 2>&1
!killall cloudflared > /dev/null 2>&1

os.system("nohup python main.py --port 8188 --listen 0.0.0.0 --highvram --dont-print-server > /tmp/comfy.log 2>&1 &")

# Chờ ComfyUI sẵn sàng
comfy_ready = False
for _ in range(40):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=1) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if not comfy_ready:
    print("⚠️ ComfyUI đang mất thêm vài giây để khởi tạo, tiếp tục mở đường hầm...")

if os.path.exists("/tmp/tunnel.log"):
    os.remove("/tmp/tunnel.log")

os.system("nohup cloudflared tunnel --url http://127.0.0.1:8188 --logfile /tmp/tunnel.log > /dev/null 2>&1 &")

public_url = None
for i in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        with open("/tmp/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

clear_output(wait=True)
if public_url:
    html_output = f"""
    <div style="background: linear-gradient(135deg, #020617, #0f172a); border: 2px solid #10b981; border-radius: 16px; padding: 24px; color: white; font-family: system-ui, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.8); max-width: 650px;">
        <div style="display:flex; align-items:center; gap:10px; margin-bottom:12px;">
            <span style="font-size:24px;">🎉</span>
            <h2 style="color: #10b981; margin: 0; font-size:20px;">COMFYUI GPU SERVER ĐÃ SẴN SÀNG!</h2>
        </div>
        <p style="font-size: 14px; color: #cbd5e1; margin-bottom: 8px;">Link kết nối GPU Cloud của bạn:</p>
        <div style="background: #000; border: 1.5px dashed #38bdf8; padding: 14px; border-radius: 10px; font-family: monospace; font-size: 16px; color: #38bdf8; font-weight: bold; word-break: break-all; margin-bottom: 16px;">
            {public_url}
        </div>
        <p style="font-size: 13px; color: #94a3b8; line-height: 1.5; margin-bottom: 0;">
            👉 Hãy copy đường link trên và dán vào <b>Cài Đặt (icon Bánh Răng) &gt; Colab GPU Server URL</b> trên Web App để bắt đầu render trực tiếp!
        </p>
    </div>
    """
    display(HTML(html_output))
    print(f"\n🔗 YOUR PUBLIC COMFYUI SERVER URL: {public_url}\n")
else:
    print("❌ Đang tạo đường hầm, vui lòng chạy lại ô này sau 5 giây!")
